# NN-VVC Phase F-3 — LIC Training on Google Colab

**Research context**: This notebook trains the Learned Image Compression (LIC) model
described in the NN-VVC paper using the 10,000-image OpenImages subset prepared in Phase F-2.

## Storage architecture

| Storage | Location | Persistent? |
|---------|----------|-------------|
| Colab local disk | `/content/` | ❌ Lost on session end |
| Google Drive | `/content/drive/MyDrive/NN_VVC/` | ✅ Permanent |

**Rule**: All checkpoints and logs are written to Google Drive.
The dataset is unzipped from Drive to local disk for I/O speed.

## Workflow
1. Detect GPU
2. Mount Google Drive
3. Configure paths
4. Verify dataset
5. Install dependencies
6. Clone / pull the repository
7. Run CPU smoke test
8. Run GPU smoke test
9. [MANUAL GATE] Start full 320-epoch training
10. Resume from latest checkpoint (if interrupted)
11. Final validation summary

## Cell 1 — GPU Detection

In [ ]:
import torch
import subprocess

print('='*60)
print('GPU Detection')
print('='*60)
print(f'CUDA available : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU name       : {gpu_name}')
    print(f'GPU memory     : {gpu_mem_gb:.1f} GB')
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=driver_version,cuda_version',
                                 '--format=csv,noheader'], capture_output=True, text=True)
        print(f'Driver / CUDA  : {result.stdout.strip()}')
    except Exception:
        pass
else:
    print('WARNING: No CUDA GPU detected — training will be VERY slow on CPU.')
    print('Go to: Runtime > Change runtime type > GPU')

print(f'PyTorch version: {torch.__version__}')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ============================================================
# Configure all paths here. Change ONLY DRIVE_ROOT if needed.
# ============================================================
DRIVE_ROOT       = Path('/content/drive/MyDrive/NN_VVC')
REPO_DIR         = Path('/content/NN_VVC')                    # local clone
LOCAL_DATA_DIR   = Path('/content/data/openimages')           # unzipped dataset

# On Drive (persistent)
DRIVE_DATASET_ZIP = DRIVE_ROOT / 'openimages_10k.zip'
DRIVE_CKPT_LIC    = DRIVE_ROOT / 'checkpoints' / 'lic'
DRIVE_CKPT_IHA    = DRIVE_ROOT / 'checkpoints' / 'iha'
DRIVE_LOGS        = DRIVE_ROOT / 'logs'

# Create Drive directories
for d in [DRIVE_ROOT, DRIVE_CKPT_LIC, DRIVE_CKPT_IHA, DRIVE_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

print('Google Drive mounted successfully.')
print(f'  Drive root      : {DRIVE_ROOT}')
print(f'  Checkpoint (LIC): {DRIVE_CKPT_LIC}')
print(f'  Logs            : {DRIVE_LOGS}')
print(f'  Dataset zip     : {DRIVE_DATASET_ZIP}')

if not DRIVE_DATASET_ZIP.exists():
    print()
    print('ERROR: Dataset archive not found on Drive.')
    print('Upload openimages_10k.zip to Google Drive at:')
    print(f'  {DRIVE_DATASET_ZIP}')
    print('Create it locally with:')
    print('  python scripts/package_f3_dataset.py pack')
    raise FileNotFoundError(f'Missing: {DRIVE_DATASET_ZIP}')

## Cell 3 — Install Dependencies

In [ ]:
# Colab already has PyTorch pre-installed.
# Install only the additional packages this project requires.
import subprocess, sys

extra_packages = [
    'scikit-image',
    'pyyaml',
    'tqdm',
]

for pkg in extra_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

print('Dependencies installed.')
import torch, torchvision
print(f'torch      : {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')

## Cell 4 — Clone / Update Repository

In [ ]:
import subprocess, os
from pathlib import Path

REPO_DIR = Path('/content/NN_VVC')
REPO_URL = 'https://github.com/Lucky-Gautam15/NN-VVC.git'  # update if needed

if REPO_DIR.exists():
    print('Repository exists — pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    print('Cloning repository...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(str(REPO_DIR))
print(f'Working directory: {os.getcwd()}')

# Add repo to Python path
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repository ready.')

## Cell 5 — Unzip Dataset from Drive

In [ ]:
from pathlib import Path

DRIVE_DATASET_ZIP = Path('/content/drive/MyDrive/NN_VVC/openimages_10k.zip')
LOCAL_DATA_DIR    = Path('/content/data/openimages')

TRAIN_DIR = LOCAL_DATA_DIR / 'train'
VAL_DIR   = LOCAL_DATA_DIR / 'val'

train_count = len(list(TRAIN_DIR.glob('*.png'))) if TRAIN_DIR.exists() else 0
val_count   = len(list(VAL_DIR.glob('*.png')))   if VAL_DIR.exists()   else 0

if train_count == 9000 and val_count == 1000:
    print(f'Dataset already extracted: {train_count} train, {val_count} val — skipping.')
else:
    print(f'Extracting dataset from {DRIVE_DATASET_ZIP} ...')
    import subprocess
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['python', 'scripts/package_f3_dataset.py', 'unpack',
         '--archive', str(DRIVE_DATASET_ZIP),
         '--output-dir', str(LOCAL_DATA_DIR)],
        check=True
    )

    train_count = len(list(TRAIN_DIR.glob('*.png')))
    val_count   = len(list(VAL_DIR.glob('*.png')))
    print(f'Extracted: {train_count} train, {val_count} val images')

## Cell 6 — Verify Dataset Manifest

In [ ]:
import json
from pathlib import Path

LOCAL_DATA_DIR = Path('/content/data/openimages')
MANIFEST_PATH  = LOCAL_DATA_DIR / 'manifest.json'

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Manifest not found: {MANIFEST_PATH}')

manifest = json.loads(MANIFEST_PATH.read_text())
images = manifest.get('images', [])
print(f'Manifest entries : {len(images)}')

train_entries = [e for e in images if e['filename'].startswith('train/')]
val_entries   = [e for e in images if e['filename'].startswith('val/')]
print(f'Train entries    : {len(train_entries)}')
print(f'Val entries      : {len(val_entries)}')

assert len(train_entries) == 9000, f'Expected 9000 train images, got {len(train_entries)}'
assert len(val_entries)   == 1000, f'Expected 1000 val images, got {len(val_entries)}'

# Quick spot-check: verify 5 random images
import hashlib, random
random.seed(42)
sample = random.sample(images, 5)
ok = 0
for entry in sample:
    fpath = LOCAL_DATA_DIR / entry['filename']
    if not fpath.exists():
        print(f'  MISSING: {fpath}')
        continue
    h = hashlib.sha256(fpath.read_bytes()).hexdigest()
    if h == entry['sha256']:
        ok += 1
    else:
        print(f'  SHA-256 MISMATCH: {fpath}')

print(f'Spot-check: {ok}/5 files verified OK')
print('\nDataset verification PASSED.')

## Cell 7 — Cache Proxy Model Weights to Drive

In [ ]:
import os
from pathlib import Path

# Point TORCH_HOME to Google Drive so Mask R-CNN weights (~170 MB)
# are downloaded once and cached permanently.
DRIVE_TORCH_CACHE = Path('/content/drive/MyDrive/NN_VVC/torch_cache')
DRIVE_TORCH_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(DRIVE_TORCH_CACHE)
print(f'TORCH_HOME set to: {DRIVE_TORCH_CACHE}')

# Pre-download Mask R-CNN weights so training does not stall mid-epoch
print('Pre-loading proxy model weights (downloads once to Drive)...')
from src.losses.proxy_loss import ProxyFeatureExtractor
_ = ProxyFeatureExtractor()  # triggers download on first run
print('Proxy model weights ready.')

## Cell 8 — CPU Smoke Test

In [ ]:
import subprocess, sys

print('Running CPU smoke test (3 epochs, 64 images, no proxy loss)...')
print('This validates the full training pipeline on CPU.')
print('=== NOT a research training run. ===')
print()

result = subprocess.run(
    [
        sys.executable, 'scripts/train_f3.py', 'lic',
        '--data-dir', '/content/data/openimages/train',
        '--val-dir',  '/content/data/openimages/val',
        '--checkpoint-dir', '/content/smoke_ckpt/lic',
        '--log-dir',  '/content/smoke_logs',
        '--epochs', '3',
        '--batch-size', '4',
        '--device', 'cpu',
        '--no-proxy',
        '--no-amp',
        '--seed', '42',
        '--num-workers', '0',
        '--run-name', 'cpu_smoke',
        '--smoke',
        '--smoke-samples', '64',
        '--smoke-epochs', '3',
    ],
    capture_output=False,
)

if result.returncode == 0:
    print('\nCPU smoke test PASSED.')
else:
    print(f'\nCPU smoke test FAILED (exit code {result.returncode}).')
    raise RuntimeError('CPU smoke test failed — fix errors before proceeding.')

## Cell 9 — GPU Smoke Test

In [ ]:
import subprocess, sys, torch

if not torch.cuda.is_available():
    print('SKIP: No CUDA GPU available.')
else:
    print('Running GPU smoke test (3 epochs, 64 images, no proxy loss)...')
    print('=== NOT a research training run. ===')
    print()

    result = subprocess.run(
        [
            sys.executable, 'scripts/train_f3.py', 'lic',
            '--data-dir', '/content/data/openimages/train',
            '--val-dir',  '/content/data/openimages/val',
            '--checkpoint-dir', '/content/smoke_ckpt_gpu/lic',
            '--log-dir',  '/content/smoke_logs_gpu',
            '--epochs', '3',
            '--batch-size', '8',
            '--device', 'cuda',
            '--no-proxy',
            '--use-amp',
            '--seed', '42',
            '--num-workers', '2',
            '--run-name', 'gpu_smoke',
            '--smoke',
            '--smoke-samples', '64',
            '--smoke-epochs', '3',
        ],
        capture_output=False,
    )

    if result.returncode == 0:
        print('\nGPU smoke test PASSED.')
    else:
        print(f'\nGPU smoke test FAILED (exit code {result.returncode}).')
        raise RuntimeError('GPU smoke test failed — fix errors before proceeding.')

## Cell 10 — Full LIC Training (320 epochs)

⚠️ **Do NOT run this cell until:**
1. CPU and GPU smoke tests pass
2. You have confirmed the dataset is correct
3. You have received explicit approval to start training

Checkpoints and logs are saved to Google Drive and will survive session restarts.

In [ ]:
# ==================================================================
# FULL TRAINING — ONLY RUN AFTER SMOKE TEST APPROVAL
# ==================================================================
# To start training, remove the `raise` line below.

raise RuntimeError(
    'FULL TRAINING GATE: Remove this line to start training.\n'
    'Only proceed after CPU and GPU smoke tests pass.'
)

import subprocess, sys

DRIVE_CKPT_LIC = '/content/drive/MyDrive/NN_VVC/checkpoints/lic'
DRIVE_LOGS     = '/content/drive/MyDrive/NN_VVC/logs'

subprocess.run(
    [
        sys.executable, 'scripts/train_f3.py', 'lic',
        '--data-dir', '/content/data/openimages/train',
        '--val-dir',  '/content/data/openimages/val',
        '--checkpoint-dir', DRIVE_CKPT_LIC,
        '--log-dir',  DRIVE_LOGS,
        '--epochs', '320',
        '--batch-size', '16',
        '--device', 'cuda',
        '--use-amp',
        '--seed', '42',
        '--num-workers', '4',
        '--max-grad-norm', '1.0',
        '--val-freq', '5',
        '--save-freq', '1',
        '--run-name', 'lic_full_320ep',
    ],
    check=True,
)

## Cell 11 — Resume from Latest Checkpoint

Run this cell if the Colab session restarted mid-training.
It finds the latest epoch checkpoint and resumes.

In [ ]:
import subprocess, sys
from pathlib import Path

DRIVE_CKPT_LIC = Path('/content/drive/MyDrive/NN_VVC/checkpoints/lic')
DRIVE_LOGS     = '/content/drive/MyDrive/NN_VVC/logs'

# Find the latest epoch checkpoint
epoch_ckpts = sorted(
    DRIVE_CKPT_LIC.glob('lic_epoch_*.pt'),
    key=lambda p: int(p.stem.split('_')[-1])
)

if not epoch_ckpts:
    print('No checkpoints found on Drive — starting from scratch.')
    resume_arg = []
else:
    latest = epoch_ckpts[-1]
    latest_epoch = int(latest.stem.split('_')[-1])
    print(f'Resuming from: {latest}  (epoch {latest_epoch})')
    resume_arg = ['--resume-from', str(latest)]

# ==================================================================
# Remove the raise below when you are ready to resume.
# ==================================================================
raise RuntimeError('Remove this line to resume training.')

subprocess.run(
    [
        sys.executable, 'scripts/train_f3.py', 'lic',
        '--data-dir', '/content/data/openimages/train',
        '--val-dir',  '/content/data/openimages/val',
        '--checkpoint-dir', str(DRIVE_CKPT_LIC),
        '--log-dir',  DRIVE_LOGS,
        '--epochs', '320',
        '--batch-size', '16',
        '--device', 'cuda',
        '--use-amp',
        '--seed', '42',
        '--num-workers', '4',
        '--run-name', 'lic_full_320ep',
    ] + resume_arg,
    check=True,
)

## Cell 12 — List Checkpoints on Drive

In [ ]:
from pathlib import Path
import torch

DRIVE_CKPT_LIC = Path('/content/drive/MyDrive/NN_VVC/checkpoints/lic')

ckpts = sorted(DRIVE_CKPT_LIC.glob('*.pt'))
print(f'Found {len(ckpts)} checkpoint(s) in {DRIVE_CKPT_LIC}')
print()

for ckpt in ckpts:
    try:
        data = torch.load(ckpt, map_location='cpu', weights_only=False)
        epoch    = data.get('epoch', '?')
        step     = data.get('step', '?')
        t_loss   = data.get('train_loss', None)
        v_loss   = data.get('val_loss', None)
        tqp      = data.get('target_qp', None)
        smoke    = data.get('config', {}).get('smoke', False) if data.get('config') else False
        tag      = ' [SMOKE]' if smoke else ''
        print(
            f'  {ckpt.name:45s}  epoch={epoch:>4}  step={step:>7}'
            f'  train_loss={t_loss:.4f}' if t_loss else '',
            f'  val_loss={v_loss:.4f}' if v_loss else '',
            f'  QP={tqp}' if tqp else '',
            tag,
        )
    except Exception as e:
        print(f'  {ckpt.name}: ERROR reading ({e})')

## Cell 13 — IHA Training (requires completed LIC training)

Only run after LIC training is complete and a QP-specific checkpoint exists.
IHA is trained per-QP (six runs: 22, 27, 32, 37, 42, 47).

In [ ]:
import subprocess, sys
from pathlib import Path

DRIVE_CKPT_LIC = Path('/content/drive/MyDrive/NN_VVC/checkpoints/lic')
DRIVE_CKPT_IHA = Path('/content/drive/MyDrive/NN_VVC/checkpoints/iha')
DRIVE_LOGS     = '/content/drive/MyDrive/NN_VVC/logs'

# Example: train IHA for QP=32 (adjust as needed)
QP = 32
LIC_CKPT = DRIVE_CKPT_LIC / f'lic_qp{QP}_epoch170.pt'

if not LIC_CKPT.exists():
    print(f'ERROR: LIC checkpoint for QP {QP} not found: {LIC_CKPT}')
    print('Complete LIC training first.')
else:
    raise RuntimeError('Remove this line to start IHA training.')

    subprocess.run(
        [
            sys.executable, 'scripts/train_f3.py', 'iha',
            '--data-dir', '/content/data/openimages/train',
            '--val-dir',  '/content/data/openimages/val',
            '--lic-checkpoint', str(LIC_CKPT),
            '--qp', str(QP),
            '--checkpoint-dir', str(DRIVE_CKPT_IHA),
            '--log-dir',  DRIVE_LOGS,
            '--epochs', '50',
            '--batch-size', '8',
            '--device', 'cuda',
            '--use-amp',
            '--seed', '42',
            '--num-workers', '4',
            '--run-name', f'iha_qp{QP}',
        ],
        check=True,
    )